In [2]:
import sys
sys.executable

'/usr/local/opt/python@3.11/bin/python3.11'

In [3]:
import ast
import pandas as pd
import numpy as np
from pygit2 import Object, Repository, GIT_SORT_TIME
from pygit2 import init_repository, Patch
from colorama import Fore
from tqdm import tqdm
import swifter
from pandarallel import pandarallel
from bs4 import BeautifulSoup, SoupStrainer
import requests
from urllib.request import urlopen
import re
import time
import random
import subprocess
import os

In [4]:
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [5]:
df_emails = pd.DataFrame()
i = 0
for file in os.listdir('data/github_commits/parquet/'):
    try:
        df_file = pd.read_parquet(f'data/github_commits/parquet/{file}')
        df_file = df_file[['commit author email', 'commmitter email', 'actor_login']].drop_duplicates()
        df_emails = pd.concat([df_emails, df_file])
        i+=1
        if i%100 == 0:
            print(i)
    except:
        print(file)

100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
.DS_Store
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
commits_pr_Azure___azure-sdk-for-python.parquet
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600


In [6]:
emails1 = df_emails['commit author email'].unique().tolist()
emails2 = df_emails['commmitter email'].unique().tolist()
emails1.extend(emails2)
emails1 = list(set(emails1))

In [15]:
emails1 = [email for email in emails1 if type(email) == str]
print(len(emails1))
print(len([email for email in emails1 if len(email.split("@")[0].split("+"))==2]))

111636
15840


In [ ]:
# I can only think of a good manual way to do this right now.... 

In [13]:
emails1

['',
 '38925938+adhiggs@users.noreply.github.com',
 'niharsalunke57@gmail.com',
 'muellesa@tf.uni-freiburg.de',
 '60656175+disalvoquentin@users.noreply.github.com',
 'timmerk@umich.edu',
 'luigi.balduzzi@imavis.com',
 'mateusz.kotas@gmail.com',
 'codingspiderfox@gmail.com',
 '130965235+shubhankarPlivo@users.noreply.github.com',
 'kfranko@google.com',
 '38914337+YajanaRao@users.noreply.github.com',
 '31191299+vdmchernov@users.noreply.github.com',
 'askrasnikov@gmail.com',
 '83480577+wx4stg@users.noreply.github.com',
 'kurt-cb@github.com',
 'alessandro.ogier@gmail.com',
 'giftangellong@gmail.com',
 '87976371+MartinaGaravaglia@users.noreply.github.com',
 'renanqts@gmail.com',
 'bkayser@bkayser-ltm1.internal.salesforce.com',
 'yaosteveliu@gmail.com',
 'pranjalbhansali@yahoo.com',
 'ye.pandaaaa906@gmail.com',
 'trackrunny@protonmail.com',
 'vladimir.serov@intel.com',
 'BrownTruck@users.noreply.github.com',
 'johnwalicki@gmail.com',
 'alex@linutronix.de',
 'brunow@element.io',
 'xiaohua.zhan

In [ ]:
pull_info = "/".join(pr_commits_url.split("/")[-3:-1]).replace("pulls","pull")
scrape_url = f"https://github.com/{repo}/{pull_info}/commits"
product = SoupStrainer('div', {'id': 'commits_bucket'})
sesh = requests.Session() 
page = sesh.get(scrape_url)
page_text = str(page.text)
if "Please wait a few minutes before you try again" in page_text:
    print('pausing, rate limit hit')
    time.sleep(120)
soup = BeautifulSoup(page.content,parse_only = product,features="html.parser")
commits = soup.find_all("a", attrs={"id":re.compile(r'commit-details*')})
commit_urls = [c['href'].split("/")[-1] for c in commits]
return commit_urls

In [61]:
def getCommits(email):
    email.replace("@","%40")
    api_url = f"https://github.com/search?q=git%40rxv.cc&type=users"
    with requests.get(api_url, auth=(username,token)) as url:
        data = url.json()
    print(data)
    if "API rate limit exceeded" in data.get('message', "no message"):
        return "pause"
    if data.get('total_count', 0) == 0:
        return []
    data_items = data['items']
    return [[d['login'], d['id'], d['type']] for d in data_items]

In [62]:
count = 0
email_data_list = []

start = time.time()
for email in emails1:
    email_data = getCommits(email)
    if email_data != "pause":
        email_data_list.append(email_data)
    count+=1

    if email_data == "pause":
        diff = time.time() - start
        sleep_time = 1 if diff > 3600 else int(3601 - diff)
        print(f"Pausing for {sleep_time} seconds after having obtained {count}") 
        time.sleep(sleep_time)
        start = time.time()
        email_data = getCommits(email)
        email_data_list.append(email_data)

{'message': 'API rate limit exceeded for user ID 33707455. If you reach out to GitHub Support for help, please include the request ID 9DDF:506C:13A9100:28A171D:65466568.', 'documentation_url': 'https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting'}
Pausing for 3600 seconds after having obtained 1


KeyboardInterrupt: 

In [60]:
pd.concat([pd.DataFrame(emails1), pd.DataFrame(email_data_list)], axis = 1).to_csv(
    'data/github_commits/csv/commit_associated_emails.csv')

,0,0
0,,None
1,whp28@cornell.edu,None
2,pineirin@gmail.com,None
3,pean.fabien@outlook.com,None
4,alexander.early@reddit.com,None
5,arrdem@alpestrine.com,None
6,git@rxv.cc,"[kpcyrd, 7763184, User]"
7,herbert.rusznak@gmail.com,None
8,pacejackson@users.noreply.github.com,None
9,60168704+parzival0611@users.noreply.github.com,None
